# 03 · Fit features inside cross-validation

Generate synthetic sequences with a known composition signal, split before
fitting, and compare four transforms inside a sklearn Pipeline. Evaluate only
the selected workflow on held-out synthetic data. This teaches leakage control;
it is **not a biological benchmark**. For real AMP MLPs use the separate ABP_GN
notebook and its local data. Requires the development checkout with sklearn,
PyWavelets, and matplotlib.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, matthews_corrcoef, roc_auc_score, RocCurveDisplay
from threadpoolctl import threadpool_limits
from protspect.sklearn import SequenceTransformer
from protspect.embeddings import kidera_factors
%matplotlib inline

In [ ]:
SEED = 7
N_SAMPLES = 120
LENGTH = 32  # Covers the generation policy, chosen before looking at test scores.
CANDIDATE_C = [0.1, 1.0]
FOLDS = 3
rng = np.random.default_rng(SEED)
sequences, labels = [], []
for i in range(N_SAMPLES):
    label = i % 2
    alphabet = list("AKKLRRV") if label else list("ADDEECV")
    sequences.append("".join(rng.choice(alphabet, size=int(rng.integers(8, 25)))))
    labels.append(label)
train, test, y_train, y_test = train_test_split(sequences, np.asarray(labels),
    test_size=.25, stratify=labels, random_state=SEED)
assert not set(train) & set(test)
print("Train / held-out sizes:", len(train), len(test))

In [ ]:
embedding = kidera_factors()
pipeline = Pipeline([
    ("features", SequenceTransformer(embedding=embedding, length=LENGTH,
        normalization="length", standardize_channels=True)),
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=SEED)),
])
options = {"fourier": {}, "z": {"radius": 1.05}, "mellin": {"mellin_c": 0.0},
    "wavelet": {"wavelet": "haar", "level": 3, "mode": "periodization"}}
grid = [{"features__method": [method], "features__transform_options": [setting],
         "model__C": CANDIDATE_C} for method, setting in options.items()]
cv = StratifiedKFold(FOLDS, shuffle=True, random_state=SEED)
search = GridSearchCV(pipeline, grid, cv=cv, scoring="balanced_accuracy",
    return_train_score=True, error_score="raise", n_jobs=1)
with threadpool_limits(limits=1):
    search.fit(train, y_train)
cv_table = pd.DataFrame(search.cv_results_)
display(cv_table[["param_features__method", "param_model__C", "mean_test_score",
                  "std_test_score", "rank_test_score"]])
print("Selected from training CV:", search.best_params_)

## Evaluate the selected model once

The validation table contains cross-validation scores, not held-out scores. The
outer synthetic test set has not selected the transform or hyperparameters.

In [ ]:
features = search.best_estimator_.named_steps["features"]
metadata_before = features.get_feature_metadata()
with threadpool_limits(limits=1):
    probabilities = search.predict_proba(test)[:, 1]
predictions = probabilities >= .5
scores = {"balanced_accuracy": balanced_accuracy_score(y_test, predictions),
          "mcc": matthews_corrcoef(y_test, predictions),
          "roc_auc": roc_auc_score(y_test, probabilities)}
display(pd.DataFrame([scores], index=["Selected model: held-out synthetic set"]))
assert metadata_before == features.get_feature_metadata()
assert json.loads(json.dumps(metadata_before)) == metadata_before
expected_mean = np.mean([embedding["KF1"][aa] for sequence in train for aa in sequence])
np.testing.assert_allclose(features.channel_mean_["KF1"], expected_mean)
print("Frozen features:", features.n_features_out_)
RocCurveDisplay.from_predictions(y_test, probabilities)
plt.title("Held-out synthetic ROC; no biological performance claim")
plt.show()

## Try another experiment

Change sample count, regularization, channel preprocessing, transform options,
or the CV seed, then rerun from the data cell. Keep the outer test set out of
selection decisions. The data were deliberately generated from different
compositions, so strong accuracy is expected and does not validate antimicrobial
prediction. Longer future chains raise rather than changing the fitted grid.
For real-data MLP comparisons, use `research/AMP_transform_MLP.ipynb`.